In [1]:
import numpy as np
from scipy.stats import uniform
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az
import os
import random
import pandas as pd

In [2]:
version = 'vr1.6.1'
file_in=f'Ge77_rates_CNP_{version}.csv'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
   
# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=['Radius[cm]','Thickness[cm]','NPanels', 'Theta[deg]', 'Length[cm]']
x_labels_out = ['Radius [cm]','Thickness [cm]','NPanels', 'Angle [deg]', 'Length [cm]']

y_label_sim = 'rGe77[nuc/(kg*yr)]'
y_label_cnp = "Ge-77_CNP"
# Set parameter boundaries
xmin=[0,0,0,0,0]
xmax=[265,20,360,90,150]

# Set parameter boundaries for aquisition function
xlow=[90,2,4,0,1]
xhigh=[250,15,360,90,150]

LF_noise = 0.028
data=pd.read_csv(f'in/{file_in}')

In [3]:
x_lf, x_hf, y_lf, y_hf = ([],[],[],[])
row_h=data.index[data['Mode'] == 1]
row_l=data.index[data['Mode'] == 0]

x_hf = data.loc[data['Mode']==1.][x_labels].to_numpy()
y_hf = data.loc[data['Mode']==1.][y_label_sim].to_numpy()

x_lf = data.loc[data['Mode']==0.][x_labels].to_numpy()
y_lf = data.loc[data['Mode']==0.][ y_label_sim].to_numpy()

In [4]:
# Generate synthetic data
np.random.seed(42)
n_samples = 100

In [5]:
from itertools import product, combinations

In [6]:
# Generate multivariate Legendre polynomial basis with interaction terms
def multivariate_legendre_with_interactions(order, x):
    """Generate multivariate Legendre polynomial basis with interactions."""
    n_samples, n_features = x.shape
    degrees = list(product(range(order + 1), repeat=n_features))
    basis = []
    for degree in degrees:
        term = np.ones(n_samples)
        for i, d in enumerate(degree):
            term *= np.polynomial.legendre.Legendre.basis(d)(x[:, i])
        basis.append(term)
    
    # Add interaction terms
    for i, j in combinations(range(n_features), 2):
        basis.append(x[:, i] * x[:, j])
    
    return np.vstack(basis).T

In [7]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

In [8]:
def cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5):
    errors = []
    kf = KFold(n_splits=5)
    for order in range(1, max_order + 1):
        phi_lf = multivariate_legendre_with_interactions(order, x_lf)
        phi_hf = multivariate_legendre_with_interactions(order, x_hf)

        # Fit LF model
        c_lf = np.linalg.lstsq(phi_lf, y_lf, rcond=None)[0]
        
        y_lf_hf_pred = phi_hf @ c_lf
        delta_hf = y_hf - y_lf_hf_pred

        mse_fold = []
        for train_idx, test_idx in kf.split(x_hf):
            x_train, x_test = x_hf[train_idx], x_hf[test_idx]
            y_train, y_test = y_hf[train_idx], y_hf[test_idx]

            phi_train = multivariate_legendre_with_interactions(order, x_train)
            phi_test = multivariate_legendre_with_interactions(order, x_test)
            
            # Bayesian fit for HF correction
            c_hf = np.linalg.lstsq(phi_train, y_train - phi_train @ c_lf, rcond=None)[0]
            y_pred_fold = phi_test @ c_hf + phi_test @ c_lf
            mse_fold.append(mean_squared_error(y_test, y_pred_fold))
        
        errors.append(np.mean(mse_fold))
    return np.argmin(errors) + 1

In [9]:
# Cross-validate for optimal polynomial order
#optimal_order = cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5)
optimal_order=1
print(f"Optimal Polynomial Order: {optimal_order}")

Optimal Polynomial Order: 1


In [10]:
from itertools import combinations_with_replacement
from numpy.polynomial.legendre import Legendre

In [11]:
def multivariate_legendre_basis(degree, x_data):
    """
    Generate the multivariate Legendre basis for multi-dimensional inputs.
    
    Parameters:
        x_data (ndarray): Input data of shape (n_samples, n_dim).
        degree (int): Maximum polynomial degree.
    
    Returns:
        basis_matrix (ndarray): Shape (n_samples, n_terms).
    """
    n_samples, n_dim = x_data.shape
    terms = []
    
    # Generate all combinations of terms up to the given degree
    for deg in range(degree + 1):
        for combo in combinations_with_replacement(range(n_dim), deg):
            terms.append(combo)

    # Evaluate each term for all samples
    basis_matrix = np.zeros((n_samples, len(terms)))
    for i, term in enumerate(terms):
        poly = np.prod([Legendre.basis(1)(x_data[:, dim]) for dim in term], axis=0)
        basis_matrix[:, i] = poly

    return basis_matrix

In [12]:
def find_indices(x_hf, x_lf):
    """
    Finds the indices of x_hf in x_lf.

    Parameters:
        x_hf (numpy.ndarray): Array of high-fidelity x values.
        x_lf (numpy.ndarray): Array of low-fidelity x values.

    Returns:
        list: Indices of x_hf in x_lf.
    """
    indices = []
    for x in x_hf:
        # Find the index of the value in x_lf that matches x_hf
        idx = np.where((x_lf == x).all(axis=1))[0]
        if idx.size > 0:
            indices.append(idx[0])  # Append the index
        else:
            raise ValueError(f"Value {x} from x_hf not found in x_lf.")
    return indices

In [13]:
def run_surrogate(x_lf,y_lf,x_hf,y_hf,x_data,y_true):
        # Define degree of PCE
        # Define degree of PCE
        degree = optimal_order
        basis_matrix_hf = multivariate_legendre_basis(degree, x_hf) # Shape: (100, 3)
        basis_matrix_lf = multivariate_legendre_basis(degree, x_lf) # Shape: (100, 3)
        indices_hf = find_indices(x_hf,x_lf)
        # Bayesian model for PCE
        # Bayesian model for PCE
        with pm.Model() as model:
            # Priors for low-fidelity coefficients
            coeffs_lf = pm.Normal("coeffs_lf", mu=0, sigma=0.3, shape=basis_matrix_lf.shape[1])
            
            # Low-fidelity predictions
            y_lf_pred_full = pm.Deterministic("y_lf_pred_full", pm.math.dot(basis_matrix_lf, coeffs_lf))
            y_lf_pred_subset = pm.Deterministic("y_lf_pred_subset", y_lf_pred_full[indices_hf])
            
            # Likelihood for low-fidelity data
            sigma_lf = pm.HalfNormal("sigma_lf", sigma=0.01)
            y_like_lf = pm.Normal("y_like_lf", mu=y_lf_pred_full, sigma=sigma_lf, observed=y_lf)
            
            # Scaling factor
            rho = pm.Normal("rho", mu=1, sigma=0.1)
            
            # Priors for high-fidelity discrepancy coefficients
            coeffs_delta = pm.Normal("coeffs_delta", mu=0, sigma=0.0005, shape=basis_matrix_hf.shape[1])
            
            # High-fidelity discrepancy
            delta_pred_from_model = pm.Deterministic("delta_pred", pm.math.dot(basis_matrix_hf, coeffs_delta))
            
            # High-fidelity predictions
            y_hf_pred = pm.Deterministic("y_hf_pred", rho * y_lf_pred_subset + delta_pred_from_model)
            
            # Likelihood for high-fidelity data
            sigma_hf = pm.HalfNormal("sigma_hf", sigma=0.005)
            y_like_hf = pm.Normal("y_like_hf", mu=y_hf_pred, sigma=sigma_hf, observed=y_hf)

            # Inference
            approx = pm.fit(n=1000000, method="advi", progressbar=True)
            trace = approx.sample(20000)  # Generate samples from the approximation

        indices_hf = np.arange(len(x_data))

        basis_matrix_hf = multivariate_legendre_basis(degree, x_data) # Shape: (100, 3)
        basis_matrix_lf = multivariate_legendre_basis(degree, x_data) # Shape: (100, 3)

        # Extract low-fidelity and high-fidelity coefficients from the posterior
        coeff_samples_lf = trace.posterior["coeffs_lf"].values  # Shape: (n_chains, n_draws, n_terms_lf)
        coeff_samples_delta = trace.posterior["coeffs_delta"].values  # Shape: (n_chains, n_draws, n_terms_hf)
        rho_samples = trace.posterior["rho"].values  # Shape: (n_chains, n_draws)

        # Flatten coefficients to combine chains and draws
        coeff_samples_lf_flat = coeff_samples_lf.reshape(-1, coeff_samples_lf.shape[-1])  # Shape: (n_samples_total, n_terms_lf)
        coeff_samples_delta_flat = coeff_samples_delta.reshape(-1, coeff_samples_delta.shape[-1])  # Shape: (n_samples_total, n_terms_hf)
        rho_samples_flat = rho_samples.flatten()  # Shape: (n_samples_total,)

        # Generate low-fidelity and discrepancy predictions
        y_lf_pred_samples = np.dot(coeff_samples_lf_flat, basis_matrix_lf.T)  # Shape: (n_samples_total, n_lf_samples)
        delta_pred_samples = np.dot(coeff_samples_delta_flat, basis_matrix_hf.T)  # Shape: (n_samples_total, n_hf_samples)

        # Compute high-fidelity predictions at high-fidelity sample locations
        y_hf_pred_samples = rho_samples_flat[:, None] * y_lf_pred_samples[:, indices_hf] + delta_pred_samples  # Shape: (n_samples_total, n_hf_samples)

        # Generate sample indices for HF samples
        hf_sample_numbers = np.arange(len(y_true))  # Shape: (n_hf_samples,)

        # Plot observed vs. predicted
        #plt.figure(figsize=(10, 6))
        #plt.fill_between(
        #    hf_sample_numbers,
        #    np.percentile(y_hf_pred_samples, 0.5, axis=0),
        #    np.percentile(y_hf_pred_samples, 99.5, axis=0),
        #    color="coral", alpha=0.2, label=r'$\pm 3\sigma$'
        #)
        #plt.fill_between(
        #    hf_sample_numbers,
        #    np.percentile(y_hf_pred_samples, 2.5, axis=0),
        #    np.percentile(y_hf_pred_samples, 97.5, axis=0),
        #    color="yellow", alpha=0.2, label=r'$\pm 2\sigma$'
        #)
        #plt.fill_between(
        #    hf_sample_numbers,
        #    np.percentile(y_hf_pred_samples, 16, axis=0),
        #    np.percentile(y_hf_pred_samples, 84, axis=0),
        #    color="green", alpha=0.2, label=r'PCE $\pm 1\sigma$'
        #)
        #plt.xlabel('HF Simulation Trial Number')
        #plt.ylabel(r'$y_{hf}$')
        #plt.scatter(hf_sample_numbers, y_true, marker='.', label="HF Validation Data", color="black")
        #handles, labels = plt.gca().get_legend_handles_labels()
        #order = [3, 2, 1, 0]
        #plt.legend([handles[idx] for idx in order], [labels[idx] for idx in order], loc=9, bbox_to_anchor=(0.665, 1.0), ncol=5)
        #plt.show()

        counter_1sigma = 0
        counter_2sigma = 0
        counter_3sigma = 0
        MSE=0
        NMSE=0
        MAE=0
        MSSE=0
        y_pred=np.mean(y_hf_pred_samples, axis=0)
        y_1sigma_low = np.percentile(y_hf_pred_samples, 16, axis=0)
        y_1sigma_high = np.percentile(y_hf_pred_samples, 84, axis=0)
        y_2sigma_low = np.percentile(y_hf_pred_samples, 2.5, axis=0)
        y_2sigma_high = np.percentile(y_hf_pred_samples, 97.5, axis=0)
        y_3sigma_low = np.percentile(y_hf_pred_samples, 0.5, axis=0)
        y_3sigma_high = np.percentile(y_hf_pred_samples, 99.5, axis=0)
        for i in range(len(y_true)):
            if (y_true[i] <= y_1sigma_high[i]) and (y_true[i] >= y_1sigma_low[i]):
                        counter_1sigma += 1

            if (y_true[i] <= y_2sigma_high[i]) and (y_true[i] >= y_2sigma_low[i]):
                        counter_2sigma += 1

            if (y_true[i] <= y_3sigma_high[i]) and (y_true[i] >= y_3sigma_low[i]):
                        counter_3sigma += 1

            percentage_1sigma=counter_1sigma/len(y_true)*100.
            percentage_2sigma=counter_2sigma/len(y_true)*100.
            percentage_3sigma=counter_3sigma/len(y_true)*100.

            MAE +=np.abs(y_true[i]-y_pred[i])
            MSE +=pow(y_true[i]-y_pred[i],2)
            y_std=y_1sigma_high[i]-y_1sigma_low[i]
            NMSE +=np.abs((y_true[i]-y_pred[i])/y_std)
            MSSE +=pow((y_true[i]-y_pred[i])/y_std,2)
        MAE=MAE/len(y_true)
        mse = mean_squared_error(y_true,y_pred, squared=True)
        NMSE=NMSE/len(y_true)
        MSSE=MSSE/len(y_true)
        MSE=MSE/len(y_true)
        means = [percentage_1sigma,percentage_2sigma,percentage_3sigma,MAE,NMSE,MSE,MSSE]
        #print(f"B-PCE & {n_samples} & 0 & {''.join([f'{x:.5f} & ' for x in means])} \\\ \hline\n")
        return means


In [14]:
%%capture
niter=100
coverage=[]

n_HF=10
sample=0
for i in range(niter):
    print('Sample #', sample)
    x_test_hf=[]
    y_test_hf=[]
    x_train_hf_sim = data.loc[data['Mode']==1.][x_labels].to_numpy().tolist()
    y_train_hf_sim = data.loc[data['Mode']==1.][y_label_sim].to_numpy().tolist()
    x_train_hf_cnp = data.loc[data['Mode']==1.][x_labels].to_numpy().tolist()
    y_train_hf_cnp = data.loc[data['Mode']==1.][y_label_sim].to_numpy().tolist()
    x_train_lf_cnp = data.loc[data['Mode']==0.][x_labels].to_numpy().tolist()
    y_train_lf_cnp = data.loc[data['Mode']==0.][y_label_sim].to_numpy().tolist()

    x_train_lf = x_train_lf_cnp.copy()
    y_train_lf = y_train_lf_cnp.copy()
    x_train_mf = x_train_hf_cnp.copy()
    y_train_mf = y_train_hf_cnp.copy()
    x_train_hf = x_train_hf_sim.copy()
    y_train_hf = y_train_hf_sim.copy()
    # Generate a list of integers from 5 to 15
    indices = random.sample(range(4, 109), 106-n_HF+4)
    for index in sorted(indices, reverse=True):
        x_train_lf.pop(300+index)
        y_train_lf.pop(300+index)
        x_train_mf.pop(index)
        y_train_mf.pop(index)
        x_train_hf.pop(index)
        y_train_hf.pop(index)
        x_test_hf.append(x_train_hf_sim[index])
        y_test_hf.append(y_train_hf_sim[index])
    x_test_hf=np.array(x_test_hf[:100])
    y_test_hf=np.array(y_test_hf[:100])
    x_train_lf=np.array(x_train_lf)
    x_train_hf=np.array(x_train_hf)
    y_train_hf=np.array(y_train_hf)
    coverage.append(run_surrogate(x_train_lf,y_train_lf,x_train_hf,y_train_hf,x_test_hf,y_test_hf))

Finished [100%]: Average Loss = -156.94
Finished [100%]: Average Loss = -159.28
Finished [100%]: Average Loss = -155.8
Finished [100%]: Average Loss = -166.23
Finished [100%]: Average Loss = -162.79
Finished [100%]: Average Loss = -162.28
Finished [100%]: Average Loss = -154.95
Finished [100%]: Average Loss = -161.24
Finished [100%]: Average Loss = -154.26
Finished [100%]: Average Loss = -149.85
Finished [100%]: Average Loss = -160.32
Finished [100%]: Average Loss = -153.26
Finished [100%]: Average Loss = -173.98
Finished [100%]: Average Loss = -158.91
Finished [100%]: Average Loss = -160.4
Finished [100%]: Average Loss = -177.75
Finished [100%]: Average Loss = -163.97
Finished [100%]: Average Loss = -148.57
Finished [100%]: Average Loss = -167.21
Finished [100%]: Average Loss = -176.02
Finished [100%]: Average Loss = -164.26
Finished [100%]: Average Loss = -152.86
Finished [100%]: Average Loss = -160.48
Finished [100%]: Average Loss = -162.83
Finished [100%]: Average Loss = -156.49
Fi

In [17]:
print("Mean:", ", ".join([f"{val:.5f}" for val in np.mean(coverage,axis=0)]))

Mean: 32.29000, 51.50000, 60.08000, 0.09629, 1.48943, 0.02028, 4.58895


In [19]:
print("Mean:", ", ".join([f"{val:.5f}" for val in np.std(coverage,axis=0)]))

Mean: 32.24757, 40.69705, 41.25401, 0.09501, 1.43884, 0.03459, 8.17526


In [18]:
print("Mean:", ", ".join([f"{val}\n" for val in coverage]))

Mean: [0.0, 0.0, 0.0, 0.20776811568480547, 3.61044067852106, 0.04591390286899408, 13.719936326199058]
, [9.0, 23.0, 51.0, 0.07622562460199425, 1.204950025468365, 0.006840254971244563, 1.651812365926418]
, [4.0, 14.000000000000002, 21.0, 0.15005644300926996, 2.0598141163223986, 0.027001667811766557, 4.8754201651439315]
, [11.0, 28.999999999999996, 50.0, 0.07600829414304354, 1.1848184348843451, 0.006900236202538432, 1.6165543286013788]
, [13.0, 65.0, 85.0, 0.054563209783651034, 0.8952963834284623, 0.003456662637627091, 0.9722591368276317]
, [43.0, 94.0, 99.0, 0.032544759689950256, 0.5302714184978836, 0.0013808654863159359, 0.3767549346884868]
, [55.00000000000001, 94.0, 99.0, 0.0312015541447594, 0.5024611275881338, 0.001256241385520365, 0.34486308908964913]
, [84.0, 95.0, 99.0, 0.017130548794768292, 0.30107899521548204, 0.0005261966068971323, 0.19964584248216585]
, [0.0, 1.0, 2.0, 0.25599623963058893, 3.0665785477057517, 0.07295192618412977, 9.85539483428122]
, [0.0, 9.0, 23.0, 0.0988292